# Práctica 4 — Calidad de Datos
**Dataset:** Enfermedades Crónicas — Pacientes Colombia  
**Curso:** Analítica de Datos 2025  

---

## 0. Instalación de dependencias

In [1]:
# Ejecutar solo la primera vez
# !pip install ydata-profiling pandas numpy matplotlib seaborn

## 1. Carga de datos originales

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('reports', exist_ok=True)
os.makedirs('models', exist_ok=True)

df_raw = pd.read_csv('data/Enfermedades_Crónicas_20260215.csv', encoding='latin-1', sep=';')

print(f'Filas: {df_raw.shape[0]}')
print(f'Columnas: {df_raw.shape[1]}')
df_raw.head()

Filas: 2602
Columnas: 29


,EDAD,PLAN_BENEFICIOS,SEDE,NOMBRE_DIAG,PESO,TALLA,IMC,CARDIOVASCULAR,PULMONAR,NEUROLÓGICO,...,covid1,covid2,covid3,GENERO,IDENTGEN,ORIGSEX,GRUPO ETNICO,NIVELEDUCATIVO,TIPODISCAPAC,OCUPACION
0,49,PACIENTES PARTICULARES,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),76,157,3083.0,Normal,Normal,Normal,...,Si,Si,No,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,COCINERA
1,49,PACIENTES PARTICULARES,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),76,157,3083.0,Normal,Normal,Normal,...,Si,Si,No,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,COCINERA
2,68,CAPITAL SALUD EPS SAS - PGP SUBSIDIADO,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),70,151,3070.0,Normal,Normal,Normal,...,Si,Si,No,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,HOGAR
3,82,CAPITAL SALUD EPS SAS - PGP SUBSIDIADO,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),51,142,2529.0,Normal,Normal,Normal,...,Si,Si,No,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,Motora-física,AMA DE CASA
4,59,CAPITAL SALUD EPS SAS - PGP SUBSIDIADO,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),84,150,3733.0,Normal,Normal,Normal,...,Si,Si,No,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,CONDUCTOR


## 2. Perfilado de datos con ydata-profiling

Se genera un reporte HTML completo con estadísticas descriptivas, distribuciones, valores nulos, correlaciones y alertas de calidad sobre los **datos originales sin modificar**.

In [3]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df_raw,
    title='Perfilado de Datos — Enfermedades Crónicas (Original)',
    explorative=True,
    correlations={
        'auto': {'calculate': True},
        'pearson': {'calculate': True},
        'spearman': {'calculate': True},
    }
)

profile.to_file('reports/profiling_report.html')
print('Reporte guardado en reports/profiling_report.html')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 77.75it/s]

Reporte guardado en reports/profiling_report.html


## 3. Diagnóstico de dimensiones de calidad de datos

Se evalúan las 6 dimensiones estándar de calidad según los resultados del perfilado.

In [4]:
print('=== RESUMEN GENERAL ===')
print(f'Total registros: {len(df_raw)}')
print(f'Total columnas: {df_raw.shape[1]}')
print()
print('=== TIPOS DE DATOS ===')
print(df_raw.dtypes)
print()
print('=== VALORES NULOS ===')
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(2)
nulos_df = pd.DataFrame({'nulos': nulos, 'porcentaje': pct_nulos})
print(nulos_df[nulos_df['nulos'] > 0])

=== RESUMEN GENERAL ===
Total registros: 2602
Total columnas: 29

=== TIPOS DE DATOS ===
EDAD                       int64
PLAN_BENEFICIOS           object
SEDE                      object
NOMBRE_DIAG               object
PESO                       int64
TALLA                      int64
IMC                      float64
CARDIOVASCULAR            object
PULMONAR                  object
NEUROLÓGICO               object
MENTAL                    object
OSTEOMUSCULAR             object
BODEX                     object
RESULTADOIMC              object
ESCALA DISNEA             object
RIESGO CARDIOVASCULAR     object
EPOCCONFIRMADO            object
CLASIFISUI                object
DISCAPACIDAD              object
covid1                    object
covid2                    object
covid3                    object
GENERO                    object
IDENTGEN                  object
ORIGSEX                   object
GRUPO ETNICO              object
NIVELEDUCATIVO            object
TIPODISCAPAC        

### 3.1 Completitud

La completitud mide la proporción de datos presentes versus esperados.

In [5]:
total_celdas = df_raw.shape[0] * df_raw.shape[1]
celdas_nulas = df_raw.isnull().sum().sum()
completitud = ((total_celdas - celdas_nulas) / total_celdas * 100).round(2)

print(f'Total celdas: {total_celdas}')
print(f'Celdas con valor nulo: {celdas_nulas}')
print(f'Completitud global: {completitud}%')
print()
print('Columnas con valores nulos:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print()
print('DIAGNÓSTICO: La completitud es muy alta (>99%). Solo la columna IMC presenta')
print('8 valores nulos (0.31%), los cuales son imputables sin afectar la calidad general.')

Total celdas: 75458
Celdas con valor nulo: 8
Completitud global: 99.99%

Columnas con valores nulos:
IMC    8
dtype: int64

DIAGNÓSTICO: La completitud es muy alta (>99%). Solo la columna IMC presenta
8 valores nulos (0.31%), los cuales son imputables sin afectar la calidad general.


### 3.2 Unicidad

La unicidad evalúa la ausencia de registros duplicados en el dataset.

In [6]:
duplicados = df_raw.duplicated().sum()
pct_dup = (duplicados / len(df_raw) * 100).round(2)

print(f'Registros duplicados: {duplicados} ({pct_dup}%)')
print(f'Registros únicos: {len(df_raw) - duplicados}')
print()
print('DIAGNÓSTICO: Se detectaron filas completamente duplicadas (32.63%). Estos registros')
print('deben eliminarse para no sesgar los modelos de minería de datos.')

if duplicados > 0:
    print()
    print('Muestra de duplicados:')
    print(df_raw[df_raw.duplicated(keep=False)].head(4))

Registros duplicados: 849 (32.63%)
Registros únicos: 1753

DIAGNÓSTICO: Se detectaron filas completamente duplicadas (32.63%). Estos registros
deben eliminarse para no sesgar los modelos de minería de datos.

Muestra de duplicados:
   EDAD                         PLAN_BENEFICIOS  \
0    49                  PACIENTES PARTICULARES   
1    49                  PACIENTES PARTICULARES   
2    68  CAPITAL SALUD EPS SAS - PGP SUBSIDIADO   
3    82  CAPITAL SALUD EPS SAS - PGP SUBSIDIADO   

                                    SEDE                       NOMBRE_DIAG  \
0  CENTRO DE SALUD TIPO II SAN CRISTÓBAL  HIPERTENSION ESENCIAL (PRIMARIA)   
1  CENTRO DE SALUD TIPO II SAN CRISTÓBAL  HIPERTENSION ESENCIAL (PRIMARIA)   
2  CENTRO DE SALUD TIPO II SAN CRISTÓBAL  HIPERTENSION ESENCIAL (PRIMARIA)   
3  CENTRO DE SALUD TIPO II SAN CRISTÓBAL  HIPERTENSION ESENCIAL (PRIMARIA)   

   PESO  TALLA     IMC CARDIOVASCULAR PULMONAR NEUROLÓGICO  ... covid1 covid2  \
0    76    157  3083.0         Normal   

### 3.3 Consistencia

La consistencia verifica que los datos sean coherentes entre sí y con las reglas de negocio.

In [7]:
# Regla 1: IMC debe ser consistente con PESO y TALLA
df_raw['IMC_calculado'] = (df_raw['PESO'] / (df_raw['TALLA'] / 100) ** 2).round(0).astype('Int64')
df_raw['IMC_int'] = df_raw['IMC'].dropna().astype(int)
inconsistencia_imc = df_raw.dropna(subset=['IMC']).eval('IMC_int != IMC_calculado').sum()
print(f'Registros con IMC inconsistente (calculado vs registrado): {inconsistencia_imc}')

# Regla 2: Edades razonables
edad_invalida = ((df_raw['EDAD'] < 0) | (df_raw['EDAD'] > 120)).sum()
print(f'Edades fuera de rango [0-120]: {edad_invalida}')

# Regla 3: Peso y talla razonables
peso_invalido = ((df_raw['PESO'] < 20) | (df_raw['PESO'] > 300)).sum()
talla_invalida = ((df_raw['TALLA'] < 100) | (df_raw['TALLA'] > 220)).sum()
print(f'Pesos fuera de rango clínico [20-300 kg]: {peso_invalido}')
print(f'Tallas fuera de rango clínico [100-220 cm]: {talla_invalida}')

# Regla 4: Valores categóricos esperados
generos_validos = ['Femenino', 'Masculino', 'No Registra', 'Intersexual']
genero_invalido = (~df_raw['GENERO'].isin(generos_validos)).sum()
print(f'Géneros con valor inesperado: {genero_invalido}')

print()
print('DIAGNÓSTICO: Se detectaron problemas de consistencia relevantes:')
print(f'  - {inconsistencia_imc} registros tienen IMC almacenado en escala ×100 (ej: 3083 en vez de 30.83).')
print(f'  - {peso_invalido} registros tienen PESO fuera del rango clínico [20-300 kg], incluyendo valores 0.')
print(f'  - {talla_invalida} registros tienen TALLA fuera del rango clínico [100-220 cm].')
print('  Estos errores serán corregidos en la etapa de limpieza.')

df_raw.drop(columns=['IMC_calculado', 'IMC_int'], inplace=True)

Registros con IMC inconsistente (calculado vs registrado): 2575
Edades fuera de rango [0-120]: 0
Pesos fuera de rango clínico [20-300 kg]: 492
Tallas fuera de rango clínico [100-220 cm]: 104
Géneros con valor inesperado: 0

DIAGNÓSTICO: Se detectaron problemas de consistencia relevantes:
  - 2575 registros tienen IMC almacenado en escala ×100 (ej: 3083 en vez de 30.83).
  - 492 registros tienen PESO fuera del rango clínico [20-300 kg], incluyendo valores 0.
  - 104 registros tienen TALLA fuera del rango clínico [100-220 cm].
  Estos errores serán corregidos en la etapa de limpieza.


### 3.4 Exactitud

La exactitud mide si los valores reflejan correctamente la realidad que representan.

In [8]:
print('Distribución de EDAD:')
print(df_raw['EDAD'].describe())
print()
print('Distribución de PESO (kg):')
print(df_raw['PESO'].describe())
print()
print('Distribución de TALLA (cm):')
print(df_raw['TALLA'].describe())
print()

ceros_peso = (df_raw['PESO'] == 0).sum()
ceros_talla = (df_raw['TALLA'] == 0).sum()
print(f'Registros con PESO = 0: {ceros_peso}')
print(f'Registros con TALLA = 0: {ceros_talla}')
print(f'Registros con PESO > 300 kg (imposible): {(df_raw["PESO"] > 300).sum()}')
print(f'Registros con IMC > 100 (escala incorrecta ×100): {(df_raw["IMC"] > 100).sum()}')
print()
print('DIAGNÓSTICO: Se detectaron errores de exactitud en variables numéricas:')
print('  - PESO tiene valores de 0 y hasta 4151 kg, indicando errores de digitación.')
print('  - IMC está almacenado en escala ×100 en la mayoría de registros (ej: 3083 = 30.83).')
print('  - TALLA presenta valores fuera de rango clínico.')
print('  Estos valores no representan mediciones reales y deben ser corregidos.')

Distribución de EDAD:
count    2602.000000
mean       68.269792
std        13.076867
min         6.000000
25%        61.000000
50%        69.000000
75%        77.000000
max       102.000000
Name: EDAD, dtype: float64

Distribución de PESO (kg):
count    2602.000000
mean      170.618755
std       328.679165
min         0.000000
25%        57.000000
50%        68.000000
75%        84.000000
max      4151.000000
Name: PESO, dtype: float64

Distribución de TALLA (cm):
count    2602.000000
mean      150.530746
std        30.022547
min         0.000000
25%       149.000000
50%       155.000000
75%       161.000000
max       300.000000
Name: TALLA, dtype: float64

Registros con PESO = 0: 17
Registros con TALLA = 0: 17
Registros con PESO > 300 kg (imposible): 429
Registros con IMC > 100 (escala incorrecta ×100): 2585

DIAGNÓSTICO: Se detectaron errores de exactitud en variables numéricas:
  - PESO tiene valores de 0 y hasta 4151 kg, indicando errores de digitación.
  - IMC está almacenado en e

### 3.5 Validez

La validez verifica que los datos cumplan el formato, tipo y dominio esperado para cada campo.

In [9]:
print('Valores únicos por columna categórica:')
cat_cols = df_raw.select_dtypes(include='object').columns
for col in cat_cols:
    n_unique = df_raw[col].nunique()
    print(f'  {col}: {n_unique} valores únicos')
    if n_unique <= 12:
        print(f'    → {df_raw[col].unique().tolist()}')

print()
print('DIAGNÓSTICO: Las columnas categóricas contienen valores dentro de dominios')
print('controlados en su mayoría. Sin embargo, se detectó un hallazgo de validez:')
print('  - PLAN_BENEFICIOS contiene el carácter corrupto \\x96 en el valor')
print('    "CAPITA SANITAS SUBSIDIADO", producto de un error de encoding en la')
print('    fuente de datos (carácter Latin-1 no convertido correctamente).')
print('  - Este valor no afecta el análisis ni el modelo, pero se documenta')
print('    como hallazgo. Se corregirá en la etapa de limpieza.')
print()
print('Los tipos numéricos (EDAD, PESO, TALLA, IMC) están correctamente tipados.')

Valores únicos por columna categórica:
  PLAN_BENEFICIOS: 8 valores únicos
    → ['PACIENTES PARTICULARES', 'CAPITAL SALUD EPS SAS - PGP SUBSIDIADO', 'EQUIPOS DE ATENCION EN CASA', 'CAPITAL SALUD EPS SAS - PGP CONTRIBUTIVO', 'CAPITAL SALUD EPS SAS - PYD SUBSIDIADO', 'COOSALUD ENTIDAD PROMOTORA DE SALUD S.A S SUBSIDIADO', 'CAPITA SANITAS SUBSIDIADO \x96 EN INTERVENCIÓN BAJO LA MEDIDA DE TOMA DE POSESIÓN', 'COOSALUD ENTIDAD PROMOTORA DE SALUD S.A.S CAPITACION']
  SEDE: 14 valores únicos
  NOMBRE_DIAG: 189 valores únicos
  CARDIOVASCULAR: 3 valores únicos
    → ['Normal', 'No Registra', 'Anormal']
  PULMONAR: 3 valores únicos
    → ['Normal', 'No Registra', 'Anormal']
  NEUROLÓGICO: 3 valores únicos
    → ['Normal', 'No Registra', 'Anormal']
  MENTAL: 3 valores únicos
    → ['Normal', 'Anormal', 'No Registra']
  OSTEOMUSCULAR: 3 valores únicos
    → ['Normal', 'No Registra', 'Anormal']
  BODEX: 1 valores únicos
    → ['MODERADA']
  RESULTADOIMC: 10 valores únicos
    → ['OBESIDAD I', 'PES

### 3.6 Oportunidad

La oportunidad evalúa si los datos están disponibles en el momento en que se necesitan y si son suficientemente recientes.

In [10]:
print('Nombre del archivo: Enfermedades_Crónicas_20260215.csv')
print('Fecha indicada en el nombre: 15 de febrero de 2026')
print()
print('DIAGNÓSTICO: El dataset fue generado recientemente (febrero 2026), lo que')
print('garantiza oportunidad. El archivo fue entregado en formato CSV estándar,')
print('accesible y procesable sin necesidad de herramientas especializadas.')
print('No hay columnas de fecha/hora que permitan evaluar la latencia de captura')
print('de cada registro individualmente.')

Nombre del archivo: Enfermedades_Crónicas_20260215.csv
Fecha indicada en el nombre: 15 de febrero de 2026

DIAGNÓSTICO: El dataset fue generado recientemente (febrero 2026), lo que
garantiza oportunidad. El archivo fue entregado en formato CSV estándar,
accesible y procesable sin necesidad de herramientas especializadas.
No hay columnas de fecha/hora que permitan evaluar la latencia de captura
de cada registro individualmente.


### Resumen del diagnóstico

In [11]:
resumen = pd.DataFrame({
    'Dimensión': ['Completitud', 'Unicidad', 'Consistencia', 'Exactitud', 'Validez', 'Oportunidad'],
    'Calificación': [
        'Alta (>99%)',
        'Baja (32.63% duplicados)',
        'Baja (IMC ×100, PESO/TALLA fuera de rango)',
        'Baja (errores de digitación en PESO e IMC)',
        'Media (carácter corrupto en PLAN_BENEFICIOS)',
        'Alta'
    ],
    'Acción requerida': [
        'Imputar 8 nulos en IMC por recálculo',
        'Eliminar 849 registros duplicados',
        'Corregir IMC ÷100, filtrar PESO y TALLA imposibles',
        'Reemplazar PESO/TALLA inválidos con mediana, recalcular IMC',
        'Reemplazar \\x96 por guión en PLAN_BENEFICIOS',
        'Sin acción'
    ]
})
resumen.style.set_caption('Diagnóstico de calidad de datos').set_table_styles(
    [{'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'white')]}]
)

,Dimensión,Calificación,Acción requerida
0,Completitud,Alta (>99%),Imputar 8 nulos en IMC por recálculo
1,Unicidad,Baja (32.63% duplicados),Eliminar 849 registros duplicados
2,Consistencia,"Baja (IMC ×100, PESO/TALLA fuera de rango)","Corregir IMC ÷100, filtrar PESO y TALLA imposibles"
3,Exactitud,Baja (errores de digitación en PESO e IMC),"Reemplazar PESO/TALLA inválidos con mediana, recalcular IMC"
4,Validez,Media (carácter corrupto en PLAN_BENEFICIOS),Reemplazar \x96 por guión en PLAN_BENEFICIOS
5,Oportunidad,Alta,Sin acción


## 4. Limpieza y mejora de datos

In [12]:
df = df_raw.copy()
print(f'Shape inicial: {df.shape}')

Shape inicial: (2602, 29)


### 4.1 Eliminación de duplicados

In [13]:
n_antes = len(df)
df = df.drop_duplicates()
n_despues = len(df)
print(f'Registros eliminados (duplicados): {n_antes - n_despues}')
print(f'Registros restantes: {n_despues}')

Registros eliminados (duplicados): 849
Registros restantes: 1753


### 4.2 Corrección de IMC en escala ×100, PESO y TALLA imposibles

In [14]:
# Primero corregir PESO y TALLA con valores imposibles
peso_invalido  = ((df['PESO'] == 0) | (df['PESO'] > 300))
talla_invalida = ((df['TALLA'] < 100) | (df['TALLA'] > 220))

print(f'Registros con PESO inválido:  {peso_invalido.sum()}')
print(f'Registros con TALLA inválida: {talla_invalida.sum()}')

df.loc[peso_invalido, 'PESO']   = np.nan
df.loc[talla_invalida, 'TALLA'] = np.nan
df['PESO']  = df['PESO'].fillna(df['PESO'].median())
df['TALLA'] = df['TALLA'].fillna(df['TALLA'].median())

# Recalcular IMC desde PESO y TALLA ya corregidos
df['IMC'] = (df['PESO'] / (df['TALLA'] / 100) ** 2).round(2)

print(f'\nDespués de corrección:')
print(f'  PESO  — min: {df["PESO"].min():.1f}, max: {df["PESO"].max():.1f}, media: {df["PESO"].mean():.1f}')
print(f'  TALLA — min: {df["TALLA"].min():.1f}, max: {df["TALLA"].max():.1f}, media: {df["TALLA"].mean():.1f}')
print(f'  IMC   — min: {df["IMC"].min():.2f}, max: {df["IMC"].max():.2f}, media: {df["IMC"].mean():.1f}')

Registros con PESO inválido:  296
Registros con TALLA inválida: 50

Después de corrección:
  PESO  — min: 1.0, max: 175.0, media: 65.9
  TALLA — min: 114.0, max: 189.0, media: 156.3
  IMC   — min: 0.41, max: 71.91, media: 27.0


### 4.3 Encoding de columnas binarias (Sí/No → 0/1)

In [15]:
# Recuperar valores originales desde df_raw usando los índices que sobrevivieron
binarias = ['DISCAPACIDAD', 'covid1', 'covid2', 'covid3']
for col in binarias:
    df[col] = df_raw.loc[df.index, col].map({'Si': 1, 'No': 0, 'No Registra': 0}).fillna(0).astype(int)
    print(f'{col}: {df[col].value_counts().to_dict()} | dtype: {df[col].dtype}')

DISCAPACIDAD: {0: 1612, 1: 141} | dtype: int64
covid1: {1: 1576, 0: 177} | dtype: int64
covid2: {1: 1531, 0: 222} | dtype: int64
covid3: {1: 918, 0: 835} | dtype: int64


### 4.4 Corrección del carácter corrupto en PLAN_BENEFICIOS

In [16]:
df['PLAN_BENEFICIOS'] = df['PLAN_BENEFICIOS'].str.replace('\x96', '-', regex=False)
df_raw['PLAN_BENEFICIOS'] = df_raw['PLAN_BENEFICIOS'].str.replace('\x96', '-', regex=False)

print('Valores únicos en PLAN_BENEFICIOS después de corrección:')
for v in df['PLAN_BENEFICIOS'].unique():
    print(f'  {v}')

Valores únicos en PLAN_BENEFICIOS después de corrección:
  PACIENTES PARTICULARES
  CAPITAL SALUD EPS SAS - PGP SUBSIDIADO
  EQUIPOS DE ATENCION EN CASA
  CAPITAL SALUD EPS SAS - PGP CONTRIBUTIVO
  CAPITAL SALUD EPS SAS - PYD SUBSIDIADO
  COOSALUD ENTIDAD PROMOTORA DE SALUD S.A S SUBSIDIADO
  CAPITA SANITAS SUBSIDIADO - EN INTERVENCIÓN BAJO LA MEDIDA DE TOMA DE POSESIÓN
  COOSALUD ENTIDAD PROMOTORA DE SALUD S.A.S CAPITACION


### 4.5 Estandarización de categorías

In [17]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].str.strip()

print('Estandarización aplicada: strip() en todas las columnas object')

Estandarización aplicada: strip() en todas las columnas object


### 4.6 Eliminación de columnas sin variabilidad

In [18]:
cols_sin_variabilidad = [col for col in df.columns if df[col].nunique() == 1]
print(f'Columnas con un solo valor único: {cols_sin_variabilidad}')
if cols_sin_variabilidad:
    df.drop(columns=cols_sin_variabilidad, inplace=True)
    print(f'Columnas eliminadas: {cols_sin_variabilidad}')

Columnas con un solo valor único: ['BODEX', 'RIESGO CARDIOVASCULAR', 'EPOCCONFIRMADO']
Columnas eliminadas: ['BODEX', 'RIESGO CARDIOVASCULAR', 'EPOCCONFIRMADO']


### 4.7 Tratamiento de outliers en variables numéricas

In [19]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['EDAD', 'PESO', 'TALLA']):
    ax.boxplot(df[col].dropna())
    ax.set_title(col)
    ax.set_ylabel('Valor')
plt.suptitle('Distribución de variables numéricas (detección de outliers)', y=1.02)
plt.tight_layout()
plt.savefig('reports/boxplot_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

for col in ['PESO', 'TALLA']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'{col}: {outliers} outliers extremos corregidos con capping IQR×3 [{lower:.1f}, {upper:.1f}]')

PESO: 23 outliers extremos corregidos con capping IQR×3 [13.0, 118.0]
TALLA: 0 outliers extremos corregidos con capping IQR×3 [114.0, 198.0]


In [20]:
# Corrección final: PESO < 20 kg es imposible para los pacientes adultos de este dataset
peso_bajo = (df['PESO'] < 20).sum()
print(f'Registros con PESO < 20 kg a corregir: {peso_bajo}')

df.loc[df['PESO'] < 20, 'PESO'] = df['PESO'].median()

# Recalcular IMC con el peso corregido
df['IMC'] = (df['PESO'] / (df['TALLA'] / 100) ** 2).round(2)

print(f'Registros con PESO < 20 kg después de corrección: {(df["PESO"] < 20).sum()}')
print(f'PESO — min: {df["PESO"].min():.1f}, max: {df["PESO"].max():.1f}, media: {df["PESO"].mean():.1f}')
print(f'IMC  — min: {df["IMC"].min():.2f}, max: {df["IMC"].max():.2f}, media: {df["IMC"].mean():.1f}')

Registros con PESO < 20 kg a corregir: 19
Registros con PESO < 20 kg después de corrección: 0
PESO — min: 21.0, max: 118.0, media: 66.5
IMC  — min: 15.06, max: 59.45, media: 27.3


### 4.8 Resultado final del dataset limpio

In [21]:
print('=== COMPARACIÓN ANTES/DESPUÉS ===')
print(f'Shape original: {df_raw.shape}')
print(f'Shape limpio:   {df.shape}')
print()
print(f'Nulos restantes: {df.isnull().sum().sum()}')
print(f'Duplicados restantes: {df.duplicated().sum()}')
print()
df.head()

=== COMPARACIÓN ANTES/DESPUÉS ===
Shape original: (2602, 29)
Shape limpio:   (1753, 26)

Nulos restantes: 0
Duplicados restantes: 0



,EDAD,PLAN_BENEFICIOS,SEDE,NOMBRE_DIAG,PESO,TALLA,IMC,CARDIOVASCULAR,PULMONAR,NEUROLÓGICO,...,covid1,covid2,covid3,GENERO,IDENTGEN,ORIGSEX,GRUPO ETNICO,NIVELEDUCATIVO,TIPODISCAPAC,OCUPACION
0,49,PACIENTES PARTICULARES,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),76.0,157.0,30.83,Normal,Normal,Normal,...,1,1,0,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,COCINERA
2,68,CAPITAL SALUD EPS SAS - PGP SUBSIDIADO,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),70.0,151.0,30.70,Normal,Normal,Normal,...,1,1,0,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,HOGAR
3,82,CAPITAL SALUD EPS SAS - PGP SUBSIDIADO,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),51.0,142.0,25.29,Normal,Normal,Normal,...,1,1,0,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,Motora-física,AMA DE CASA
4,59,CAPITAL SALUD EPS SAS - PGP SUBSIDIADO,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),84.0,150.0,37.33,Normal,Normal,Normal,...,1,1,0,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,CONDUCTOR
5,57,CAPITAL SALUD EPS SAS - PGP SUBSIDIADO,CENTRO DE SALUD TIPO II SAN CRISTÓBAL,HIPERTENSION ESENCIAL (PRIMARIA),65.0,162.0,24.77,Normal,Normal,Normal,...,1,1,0,Femenino,Cisgénero,Heterosexual,Ninguna de las anteriores,14. Sin Informacion,No Presenta,OFICIOS VARIOS


In [22]:
df.to_csv('data/chronic_disease_clean.csv', index=False, encoding='utf-8')
print('Dataset limpio guardado en data/chronic_disease_clean.csv')

Dataset limpio guardado en data/chronic_disease_clean.csv
